### Importing Dependencies

In [1]:
# Import randint and shuffle functions from the random library
from random import randint as rnd
from random import shuffle


### Setting The Problem and Algorithms Parameters

In [2]:
N = 7  # Number of items
MAX_WEIGHT = 14  # Maximum bag weight
objects = [(10, 2), (5, 3), (15, 5), (7, 7), (6, 1), (18, 4), (3, 1)]  # List of objects (value, weight)

POPULAION_SIZE = 200  # Population size
MUTATION_RATE = 0.8  # Mutation rate

EPOCH = 100  # Number of generations


### Item Class

In [3]:
# Define the Item class to store profit and weight information
class Item:
    def __init__(self, profit, weight):
        self.profit = profit
        self.weight = weight


### Getting The Items Properties

In [4]:
# A function to get input (items) or use predefined input values
def get_input(n, input_items = None, verbose=0):
    items = []  # A list to store items
    if items==None:  # If no input is provided, take input from the user
        for i in range(n):
            print(f"Item #{i+1}")
            item_profit = int(input("What is the profit of this item? "))
            item_weigth = int(input("What is the weigth of this item? "))
            items.append(Item(item_profit, item_weigth))  # Create the item and add it to the list
        print("###########################################################")
    else:  # If the input is provided as a list, use it
        for item in input_items:
            items.append(Item(item[0], item[1]))  # Create an item from the list and add it to the items list
    if verbose:  # If verbose mode is enabled, print item details
        for item in items:
            print(f"Item #{items.index(item)+1}: weight:{item.weight}  profit:{item.profit}")
    return items  # Return the list of items


### Initial Population Function

In [5]:
# A function for initializing the population
def init_population(n, p):
    population_list = []  # Population list
    for i in range(p):
        new_member = [0 for i in range(n)] + [1 for i in range(n)]  # Create a chromosome with a mix of 0s and 1s of length n, plus two extra slots for profit and weight
        shuffle(new_member)  # Randomly shuffle the chromosome
        new_member = new_member[:n] + [None, None]  # Add two empty slots for profit and weight
        population_list.append(new_member)  # Add the chromosome to the population list
    return population_list  # Return the population


### Cross Over Function

In [6]:
# A function for crossover (combining two parents to create offspring)
def cross_over(population_list, n, p):
    for i in range(0, p, 2):
        child1 = population_list[i][:n//2] + population_list[i+1][n//2:n] + [None, None]  # First child: first half of parent 1 and second half of parent 2
        child2 = population_list[i+1][:n//2] + population_list[i][n//2:n] + [None, None]  # Second child: first half of parent 2 and second half of parent 1
        # Add offspring to the population
        population_list.append(child1)
        population_list.append(child2)
    return population_list  # Return the population containing new offspring


### Mutation Function

In [7]:
# Mutation function (randomly changing genes)
def mutation(population_list, n, p, m):
    chossen_ones = [i for i in range(p, p*2)] # Select individuals for mutation
    shuffle(chossen_ones) # Randomize the selection
    chossen_ones = chossen_ones[:int(((p*2)-1)*m)]  # Select a specific number of individuals based on mutation rate
    for i in chossen_ones:
        cell = rnd(0, n-1)  # Choose a random gene index
        population_list[i][cell] = 1 if population_list[i][cell] == 0 else 0  # Flip the gene (0 to 1 or vice versa)
    return population_list  # Return the population after mutation


### Fitness Function

In [8]:
# A function to calculate the weight difference between the knapsack and the maximum weight capacity
def weight_distance(bag, n, max_weight, items_list):
    total_weight = 0  # Total weight of the knapsack
    for i in range(n):
        if bag[i]:  # If the item is selected (chromosome value is 1)
            total_weight+=items_list[i].weight  # Add the item's weight to the total weight
    # If the total weight exceeds the maximum capacity, return the difference; otherwise, return 200
    return abs(max_weight-total_weight) if total_weight>max_weight else 200  
  
# A function to calculate the total profit of the knapsack
def profit(bag, n, items_list):
    total_profit = 0  # Total profit
    for i in range(n):
        if bag[i]:  # If the item is selected
            total_profit+=items_list[i].profit  # Add the item's profit to the total profit
    return total_profit  # Return the total profit

# A function to evaluate and assign fitness values to each chromosome
def fitness(population_list, n, p, items_list, max_weight):
    for i in range(p*2):
        if population_list[i][n]==None or population_list[i][n+1]==None:
            # Calculate the weight distance and profit for chromosomes that have not been evaluated yet
            population_list[i][n]= weight_distance(population_list[i], n, max_weight, items_list)
            population_list[i][n+1]= profit(population_list[i], n, items_list)
    return population_list  # Return the population with updated fitness values


In [9]:
# A function to sort the population based on fitness (first by weight, then by profit)
def sorter(population_list, index1, index2):
    sorted_list = sorted(population_list, key=lambda x: (x[index1], -x[index2]))  # Sort the population
    return sorted_list  # Return the sorted population


### Main

In [10]:
# Main execution block
if __name__ == "__main__":
    print(f"Max Weight is {MAX_WEIGHT}")
    print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
    # Get the items
    items = get_input(N, input_items=objects, verbose=1)
    print("#######################################################")
    # Initialize the population
    current_population = init_population(N, POPULAION_SIZE)
    while EPOCH:  # Run the loop until all generations are processed
        current_population = cross_over(current_population, N, POPULAION_SIZE)  # Crossover
        current_population = mutation(current_population, N, POPULAION_SIZE, MUTATION_RATE)  # Mutation
        current_population = fitness(current_population, N, POPULAION_SIZE, items, MAX_WEIGHT)  # Calculate fitness
        current_population = sorter(current_population, N, N+1)  # Sort the population
        current_population = current_population[:POPULAION_SIZE]  # Reduce population to its initial size
        EPOCH -= 1  # Decrease the remaining generation count
    else:
        print("Best Found Solution: ")
        for i in current_population:
            print(i)  # Display the best found solutions


Max Weight is 14
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
Item #1: weight:2  profit:10
Item #2: weight:3  profit:5
Item #3: weight:5  profit:15
Item #4: weight:7  profit:7
Item #5: weight:1  profit:6
Item #6: weight:4  profit:18
Item #7: weight:1  profit:3
#######################################################
Best Found Solution: 
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0, 1, 1, 0, 1, 54]
[1, 1, 1, 0,